# Tool-Based Agent Pattern: MCP
### When tools need their own runtime — standardized, composable, and isolated

The agent **delegates**. The server **executes**. The MCP boundary is your governance checkpoint — audit, auth, and rate limiting live at the server layer.

Use MCP when a process operation involves:
- **Complexity** — long-running computation or multi-step transforms
- **Stateful execution** — sessions, connections, or transaction context
- **Enterprise system integration** — connecting to external APIs behind a governed boundary

### Business Use Case: AnyComp Telecom Network Operations Center

The NOC team needs an agent that can look up trouble tickets, check device status, and run network diagnostics via an MCP server.

### Architecture

```
User Query → Agent ─── MCP Protocol ───→ MCP Server (NOC Mock)
                                              │
                                    ┌─────────┼────────────┐
                                    ▼         ▼            ▼
                              get_ticket  check_device  run_diagnostic
                                    │         │            │
                                    └─────────┼────────────┘
                                              ▼
             Agent ←─── MCP Protocol ─── Structured result
                │
         Reason over results → Response
```



## Install Dependencies

Run this cell first, then **restart the kernel** and continue.

In [ ]:
!pip install -q strands-agents>=1.36.0 strands-agents-tools boto3 "typing_extensions>=4.12.0" mcp

## Setup

Configure the path to the NOC mock MCP server. The server exposes three tools:
- `get_ticket` — look up a trouble ticket
- `check_device_status` — check if a network device is healthy/degraded/down
- `run_diagnostic` — run a ping, throughput, or signal test

In [ ]:
import boto3
import sys
import os
from pathlib import Path

# Locate the MCP server
# Find common dir — handle different kernel working directories
COMMON_DIR = None
for candidate in [Path('../common'), Path('common'), Path('../../agent-orchestration-patterns/common')]:
    if candidate.exists():
        COMMON_DIR = candidate.resolve()
        break
assert COMMON_DIR, "common/ directory not found"
MCP_SERVER = str(COMMON_DIR / 'mcp_servers' / 'noc_mock' / 'server.py')
MOCK_DATA = str(COMMON_DIR / 'mcp_servers' / 'noc_mock' / 'mock_data.json')

REGION = boto3.session.Session().region_name or "us-east-1"

print(f'MCP Server: {MCP_SERVER}')
print(f'Mock Data : {MOCK_DATA}')

## Sample Request

In [ ]:
REQUEST = "Look up trouble ticket TKT-4001. Check the status of the device associated with it. Then run a ping diagnostic and a throughput diagnostic on that device. Give me a summary of the situation and your recommendation."

## Strands Implementation

Strands has native MCP support via `MCPClient`. Tools are auto-discovered from the server — no manual wrapping needed.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

noc_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command=sys.executable,
            args=[MCP_SERVER],
            env={"MOCK_DATA_PATH": MOCK_DATA, **os.environ},
        )
    )
)

strands_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-6",
    region_name=REGION,
    max_tokens=4096,
)

with noc_mcp_client:
    mcp_tools = noc_mcp_client.list_tools_sync()
    print(f"Discovered {len(mcp_tools)} MCP tools: {[t.tool_name for t in mcp_tools]}")
    print()

    strands_agent = Agent(
        model=strands_model,
        system_prompt=(
            "You are a Network Operations Center agent for AnyComp Telecom. "
            "Use your MCP tools to investigate trouble tickets, check device status, "
            "and run diagnostics. Provide clear summaries and actionable recommendations."
        ),
        tools=mcp_tools,
    )

    print("=" * 60)
    print("STRANDS AGENTS (MCP)")
    print("=" * 60)
    print(strands_agent(REQUEST))

## Function Calling vs MCP: When to Use Which

| | Function Calling | MCP |
|---|---|---|
| **Tool runs in** | Same Python process as the agent | Separate server process |
| **Best for** | Calculations, lookups, rule enforcement | Enterprise systems, stateful ops, long-running tasks |
| **Governance** | Inline — trust the agent’s process | Server boundary — audit, auth, rate limiting at the server |
| **Composability** | Tied to the agent’s codebase | Any MCP-compliant agent speaks to any MCP-compliant server |
| **Isolation** | Shared memory, shared failures | Separate runtime, independent scaling |

Architecture principle: the MCP boundary is your governance checkpoint. Complex logic, external connections, and long-running operations run in their own context.